# Hurricane Melissa NDVI Recovery To Pre-Event Baseline

This notebook treats the months 1-2 HLS composite as the early post-event damage state, not as a later recovery period. It identifies benefit-providing NbS pixels with a relative NDVI decline of at least 10% in months 1-2 compared with the pre-event baseline, then evaluates recovery in months 3-4 and months 5-6.

Recovery is reported in two complementary ways:

- **Recovery fraction of initial loss**: `(NDVI_window - NDVI_months_1_2) / (NDVI_pre_event - NDVI_months_1_2)`. A value of 90% means the pixel has regained 90% of the NDVI loss observed in months 1-2.
- **Current NDVI relative to pre-event baseline**: `NDVI_window / NDVI_pre_event`. A value of 90% means the pixel's current NDVI is at least 90% of its pre-event NDVI.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from rasterio.features import rasterize
from rasterio.warp import Resampling, reproject

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.6,
    }
)

## Paths And Constants

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
paper2_path = base_path / "dphil_paper_2"
paper3_path = base_path / "dphil_paper_3"

output_dir = (
    paper3_path
    / "results"
    / "threats"
    / "hurricane_melissa_damage"
    / "recovery"
    / "recovery_to_pre_event_baseline"
)
output_dir.mkdir(parents=True, exist_ok=True)

mangrove_patches_path = paper3_path / "inputs" / "forces_of_nature_mangroves" / "mangroves.shp"
mangrove_patch_table_path = (
    paper3_path
    / "results"
    / "threats"
    / "hurricane_melissa_damage"
    / "mangrove_eads_hurricane_damage"
    / "mangrove_ead_hurricane_damage_patch_table.csv"
)
river_ead_min_path = paper2_path / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_min.tif"
river_ead_max_path = paper2_path / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_max.tif"

ndvi_window_paths = {
    "before": paper3_path / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif",
    "months_1_2": paper3_path / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif",
    "months_3_4": paper3_path / "inputs" / "ndvi" / "HLS_masked_NDVI_months3to4_after_epsg3448_2025-12-30_to_2026-02-28.tif",
    "months_5_6": paper3_path / "inputs" / "ndvi" / "HLS_masked_NDVI_months5to6_after_epsg3448_2026-03-01_to_2026-04-29.tif",
}
ndvi_window_labels = {
    "before": "Pre-event",
    "months_1_2": "Months 1-2",
    "months_3_4": "Months 3-4",
    "months_5_6": "Months 5-6",
}
ndvi_time_order = ["before", "months_1_2", "months_3_4", "months_5_6"]
recovery_windows = ["months_3_4", "months_5_6"]

jmd_to_usd = 1.0 / 150.0
relative_baseline_min = 0.20
relative_damage_threshold = -0.10
figure_dpi = 300

required_paths = [
    mangrove_patches_path,
    mangrove_patch_table_path,
    river_ead_min_path,
    river_ead_max_path,
    *ndvi_window_paths.values(),
]
for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

output_dir

## Helper Functions

In [ ]:
def valid_ndvi(ndvi_array: np.ndarray) -> np.ndarray:
    """Return valid NDVI cells on the expected -1 to 1 range."""
    return np.isfinite(ndvi_array) & (ndvi_array >= -1.0) & (ndvi_array <= 1.0)


def percentage(numerator: float, denominator: float) -> float:
    """Return percentage, preserving NaN when the denominator is zero."""
    return float(numerator / denominator * 100.0) if denominator else np.nan


def finite_mean(values: np.ndarray) -> float:
    """Return the mean of finite values, preserving NaN for empty inputs."""
    finite_values = values[np.isfinite(values)]
    return float(np.mean(finite_values)) if finite_values.size else np.nan


def finite_median(values: np.ndarray) -> float:
    """Return the median of finite values, preserving NaN for empty inputs."""
    finite_values = values[np.isfinite(values)]
    return float(np.median(finite_values)) if finite_values.size else np.nan


def sum_positive_values(value_array: np.ndarray, mask: np.ndarray) -> float:
    """Sum finite positive raster values inside a mask."""
    selected_values = value_array[mask]
    finite_positive_values = selected_values[np.isfinite(selected_values) & (selected_values > 0)]
    return float(np.sum(finite_positive_values)) if finite_positive_values.size else 0.0


def read_aligned_ndvi(window_paths: dict[str, Path]) -> tuple[dict[str, np.ndarray], dict]:
    """Read NDVI rasters and verify they share one grid."""
    ndvi_arrays = {}
    reference_profile = None
    for window_name, window_path in window_paths.items():
        with rasterio.open(window_path) as source_raster:
            profile = source_raster.profile.copy()
            ndvi_array = source_raster.read(1).astype("float32")
        if reference_profile is None:
            reference_profile = profile
        else:
            alignment_checks = {
                "crs": profile["crs"] == reference_profile["crs"],
                "transform": profile["transform"] == reference_profile["transform"],
                "height": profile["height"] == reference_profile["height"],
                "width": profile["width"] == reference_profile["width"],
            }
            if not all(alignment_checks.values()):
                raise ValueError(f"Raster alignment mismatch for {window_path}: {alignment_checks}")
        ndvi_arrays[window_name] = ndvi_array
    return ndvi_arrays, reference_profile


def read_positive_ead_usd(path: Path, reference_profile: dict | None = None) -> tuple[np.ndarray, dict]:
    """Read avoided EAD raster, convert JMD to USD, and keep only positive cells."""
    with rasterio.open(path) as source_raster:
        profile = source_raster.profile.copy()
        ead_array = source_raster.read(1).astype("float64") * jmd_to_usd
    ead_array[~np.isfinite(ead_array) | (ead_array <= 0)] = np.nan
    if reference_profile is not None:
        alignment_checks = {
            "crs": profile["crs"] == reference_profile["crs"],
            "transform": profile["transform"] == reference_profile["transform"],
            "height": profile["height"] == reference_profile["height"],
            "width": profile["width"] == reference_profile["width"],
        }
        if not all(alignment_checks.values()):
            raise ValueError(f"Raster alignment mismatch for {path}: {alignment_checks}")
    return ead_array, profile


def reproject_ndvi_to_reference(path: Path, reference_profile: dict) -> np.ndarray:
    """Reproject a continuous NDVI raster to the river restoration-benefit grid."""
    destination = np.full(
        (reference_profile["height"], reference_profile["width"]),
        np.nan,
        dtype="float32",
    )
    with rasterio.open(path) as source_raster:
        reproject(
            source=rasterio.band(source_raster, 1),
            destination=destination,
            src_transform=source_raster.transform,
            src_crs=source_raster.crs,
            src_nodata=source_raster.nodata,
            dst_transform=reference_profile["transform"],
            dst_crs=reference_profile["crs"],
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return destination


def recovery_fraction(initial_values: np.ndarray, early_values: np.ndarray, window_values: np.ndarray) -> np.ndarray:
    """Return fraction of the early post-event NDVI loss recovered by a later window."""
    initial_drop = initial_values - early_values
    fraction = np.full(initial_values.shape, np.nan, dtype="float64")
    positive_drop = initial_drop > 0
    fraction[positive_drop] = (window_values[positive_drop] - early_values[positive_drop]) / initial_drop[positive_drop]
    return fraction


def recovery_category_rows(
    ecosystem: str,
    window_name: str,
    fraction_values: np.ndarray,
    pixel_area_ha: float,
) -> list[dict]:
    """Classify damaged pixels by recovery fraction category."""
    valid_fraction = np.isfinite(fraction_values)
    total_pixels = int(valid_fraction.sum())
    categories = [
        ("No improvement or further decline", fraction_values <= 0.0),
        ("0-<50% of initial loss recovered", (fraction_values > 0.0) & (fraction_values < 0.5)),
        ("50-<90% of initial loss recovered", (fraction_values >= 0.5) & (fraction_values < 0.9)),
        ("90-<100% of initial loss recovered", (fraction_values >= 0.9) & (fraction_values < 1.0)),
        (">=100% of initial loss recovered", fraction_values >= 1.0),
    ]
    rows = []
    for category_name, category_mask in categories:
        pixel_count = int((valid_fraction & category_mask).sum())
        rows.append(
            {
                "ecosystem": ecosystem,
                "window": window_name,
                "window_label": ndvi_window_labels[window_name],
                "category": category_name,
                "pixel_count": pixel_count,
                "area_ha": pixel_count * pixel_area_ha,
                "share_of_damaged_area_pct": percentage(pixel_count, total_pixels),
            }
        )
    return rows


def summarize_recovery(
    ecosystem: str,
    benefit_mask: np.ndarray,
    ndvi_arrays: dict[str, np.ndarray],
    pixel_area_ha: float,
    ead_min_usd: np.ndarray | None = None,
    ead_max_usd: np.ndarray | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Summarize early damage and later recovery for one benefit-providing NbS type."""
    valid_masks = {window_name: valid_ndvi(ndvi_array) for window_name, ndvi_array in ndvi_arrays.items()}
    full_series_mask = benefit_mask.copy()
    for valid_mask in valid_masks.values():
        full_series_mask &= valid_mask

    baseline_eligible_mask = full_series_mask & (ndvi_arrays["before"] >= relative_baseline_min)
    relative_change_early = np.full(benefit_mask.shape, np.nan, dtype="float32")
    relative_change_early[baseline_eligible_mask] = (
        ndvi_arrays["months_1_2"][baseline_eligible_mask]
        - ndvi_arrays["before"][baseline_eligible_mask]
    ) / ndvi_arrays["before"][baseline_eligible_mask]
    damaged_mask = baseline_eligible_mask & (relative_change_early <= relative_damage_threshold)

    total_minimum_ead = sum_positive_values(ead_min_usd, np.isfinite(ead_min_usd)) if ead_min_usd is not None else np.nan
    total_maximum_ead = sum_positive_values(ead_max_usd, np.isfinite(ead_max_usd)) if ead_max_usd is not None else np.nan
    damaged_minimum_ead = sum_positive_values(ead_min_usd, damaged_mask) if ead_min_usd is not None else np.nan
    damaged_maximum_ead = sum_positive_values(ead_max_usd, damaged_mask) if ead_max_usd is not None else np.nan

    coverage_summary = pd.DataFrame(
        [
            {
                "ecosystem": ecosystem,
                "benefit_area_full_series_ha": int(full_series_mask.sum()) * pixel_area_ha,
                "baseline_eligible_area_ha": int(baseline_eligible_mask.sum()) * pixel_area_ha,
                "early_post_event_damaged_area_ha": int(damaged_mask.sum()) * pixel_area_ha,
                "pct_full_series_area_damaged_early_post_event": percentage(damaged_mask.sum(), full_series_mask.sum()),
                "positive_avoided_ead_usd_minimum_damaged": damaged_minimum_ead,
                "positive_avoided_ead_usd_maximum_damaged": damaged_maximum_ead,
                "pct_total_positive_avoided_ead_minimum_damaged": percentage(damaged_minimum_ead, total_minimum_ead),
                "pct_total_positive_avoided_ead_maximum_damaged": percentage(damaged_maximum_ead, total_maximum_ead),
            }
        ]
    )

    trajectory_rows = []
    for window_name in ndvi_time_order:
        trajectory_rows.append(
            {
                "ecosystem": ecosystem,
                "window": window_name,
                "window_label": ndvi_window_labels[window_name],
                "damaged_pixel_count": int(damaged_mask.sum()),
                "damaged_area_ha": int(damaged_mask.sum()) * pixel_area_ha,
                "mean_ndvi_damaged_pixels": finite_mean(ndvi_arrays[window_name][damaged_mask]),
                "median_ndvi_damaged_pixels": finite_median(ndvi_arrays[window_name][damaged_mask]),
            }
        )
    trajectory_summary = pd.DataFrame(trajectory_rows)

    headline_rows = []
    category_rows = []
    before_values = ndvi_arrays["before"][damaged_mask]
    early_values = ndvi_arrays["months_1_2"][damaged_mask]
    initial_drop = before_values - early_values
    for window_name in recovery_windows:
        window_values = ndvi_arrays[window_name][damaged_mask]
        fraction_values = recovery_fraction(before_values, early_values, window_values)
        current_pct_pre_event = np.full(before_values.shape, np.nan, dtype="float64")
        positive_before = before_values > 0
        current_pct_pre_event[positive_before] = window_values[positive_before] / before_values[positive_before] * 100.0
        clipped_fraction_pct = np.clip(fraction_values, 0.0, 1.0) * 100.0
        headline_rows.append(
            {
                "ecosystem": ecosystem,
                "window": window_name,
                "window_label": ndvi_window_labels[window_name],
                "damaged_pixel_count": int(damaged_mask.sum()),
                "damaged_area_ha": int(damaged_mask.sum()) * pixel_area_ha,
                "mean_pre_event_ndvi": finite_mean(before_values),
                "mean_months_1_2_ndvi": finite_mean(early_values),
                "mean_window_ndvi": finite_mean(window_values),
                "mean_initial_drop_ndvi": finite_mean(initial_drop),
                "mean_recovered_ndvi_since_months_1_2": finite_mean(window_values - early_values),
                "mean_remaining_gap_to_pre_event_ndvi": finite_mean(before_values - window_values),
                "mean_recovery_fraction_of_initial_loss_pct": finite_mean(fraction_values * 100.0),
                "median_recovery_fraction_of_initial_loss_pct": finite_median(fraction_values * 100.0),
                "mean_clipped_recovery_fraction_of_initial_loss_pct": finite_mean(clipped_fraction_pct),
                "mean_current_ndvi_pct_of_pre_event": finite_mean(current_pct_pre_event),
                "median_current_ndvi_pct_of_pre_event": finite_median(current_pct_pre_event),
                "pct_improved_vs_months_1_2": percentage(np.sum(window_values > early_values), window_values.size),
                "pct_recovered_at_least_50pct_initial_loss": percentage(np.sum(fraction_values >= 0.5), fraction_values.size),
                "pct_recovered_at_least_90pct_initial_loss": percentage(np.sum(fraction_values >= 0.9), fraction_values.size),
                "pct_recovered_to_or_above_pre_event": percentage(np.sum(window_values >= before_values), window_values.size),
                "pct_current_ndvi_at_least_90pct_pre_event": percentage(np.sum(window_values >= before_values * 0.9), window_values.size),
                "pct_still_below_pre_event": percentage(np.sum(window_values < before_values), window_values.size),
            }
        )
        category_rows.extend(recovery_category_rows(ecosystem, window_name, fraction_values, pixel_area_ha))

    return coverage_summary, pd.DataFrame(headline_rows), pd.DataFrame(category_rows), trajectory_summary


def save_figure(figure: plt.Figure, stem: str) -> list[Path]:
    """Save one figure to PNG, SVG, and PDF."""
    output_paths = []
    for suffix in ["png", "svg", "pdf"]:
        output_path = output_dir / f"{stem}.{suffix}"
        figure.savefig(output_path, dpi=figure_dpi, bbox_inches="tight")
        output_paths.append(output_path)
    return output_paths

## Mangrove Benefit Pixels

In [ ]:
mangrove_ndvi_arrays, mangrove_ndvi_profile = read_aligned_ndvi(ndvi_window_paths)
mangrove_pixel_area_ha = abs(mangrove_ndvi_profile["transform"].a * mangrove_ndvi_profile["transform"].e) / 10_000.0

mangrove_patches = gpd.read_file(mangrove_patches_path).to_crs(mangrove_ndvi_profile["crs"])
mangrove_patch_table = pd.read_csv(mangrove_patch_table_path)
positive_mangrove_ids = set(
    mangrove_patch_table.loc[
        mangrove_patch_table["positive_avoided_ead_either"].fillna(False).astype(bool),
        "Mangrove_ID",
    ].astype(int)
)
mangrove_patches["Mangrove_ID"] = mangrove_patches["ID"].astype(int)
benefit_mangrove_patches = mangrove_patches[mangrove_patches["Mangrove_ID"].isin(positive_mangrove_ids)].copy()
benefit_mangrove_shapes = [(geometry, 1) for geometry in benefit_mangrove_patches.geometry]
benefit_mangrove_mask = rasterize(
    benefit_mangrove_shapes,
    out_shape=(mangrove_ndvi_profile["height"], mangrove_ndvi_profile["width"]),
    transform=mangrove_ndvi_profile["transform"],
    fill=0,
    dtype="uint8",
).astype(bool)

print(f"Benefit-providing mangrove patches: {len(benefit_mangrove_patches):,}")
print(f"Rasterized mangrove benefit area: {benefit_mangrove_mask.sum() * mangrove_pixel_area_ha:,.1f} ha")

## River-Flood Forest Restoration Benefit Pixels

In [ ]:
river_ead_min_usd, river_profile = read_positive_ead_usd(river_ead_min_path)
river_ead_max_usd, _ = read_positive_ead_usd(river_ead_max_path, river_profile)
river_pixel_area_ha = abs(river_profile["transform"].a * river_profile["transform"].e) / 10_000.0
river_benefit_mask = np.isfinite(river_ead_min_usd) | np.isfinite(river_ead_max_usd)
# HLS NDVI is read again intentionally here because river avoided-EAD pixels use a different grid from the native HLS/mangrove grid.
river_ndvi_arrays = {
    window_name: reproject_ndvi_to_reference(window_path, river_profile)
    for window_name, window_path in ndvi_window_paths.items()
}

print(f"River-flood restoration benefit area: {river_benefit_mask.sum() * river_pixel_area_ha:,.1f} ha")
print(
    "Positive avoided EAD: "
    f"US${sum_positive_values(river_ead_min_usd, np.isfinite(river_ead_min_usd)) / 1e6:,.2f}-"
    f"{sum_positive_values(river_ead_max_usd, np.isfinite(river_ead_max_usd)) / 1e6:,.2f} million"
)

## Recovery Summaries

In [ ]:
mangrove_coverage, mangrove_headline, mangrove_categories, mangrove_trajectory = summarize_recovery(
    "Mangroves",
    benefit_mangrove_mask,
    mangrove_ndvi_arrays,
    mangrove_pixel_area_ha,
)
forest_coverage, forest_headline, forest_categories, forest_trajectory = summarize_recovery(
    "River forest restoration",
    river_benefit_mask,
    river_ndvi_arrays,
    river_pixel_area_ha,
    river_ead_min_usd,
    river_ead_max_usd,
)

coverage_summary = pd.concat([mangrove_coverage, forest_coverage], ignore_index=True)
headline_summary = pd.concat([mangrove_headline, forest_headline], ignore_index=True)
category_summary = pd.concat([mangrove_categories, forest_categories], ignore_index=True)
trajectory_summary = pd.concat([mangrove_trajectory, forest_trajectory], ignore_index=True)

coverage_summary

In [ ]:
display_columns = [
    "ecosystem",
    "window_label",
    "damaged_area_ha",
    "mean_pre_event_ndvi",
    "mean_months_1_2_ndvi",
    "mean_window_ndvi",
    "mean_recovery_fraction_of_initial_loss_pct",
    "median_recovery_fraction_of_initial_loss_pct",
    "mean_clipped_recovery_fraction_of_initial_loss_pct",
    "mean_current_ndvi_pct_of_pre_event",
    "pct_recovered_at_least_90pct_initial_loss",
    "pct_current_ndvi_at_least_90pct_pre_event",
    "pct_recovered_to_or_above_pre_event",
    "pct_still_below_pre_event",
]
display(headline_summary[display_columns].round(2))

In [ ]:
display(category_summary.round(2))

## Figures

In [ ]:
trajectory_plot = trajectory_summary.copy()
trajectory_plot["window_label"] = pd.Categorical(
    trajectory_plot["window_label"],
    categories=[ndvi_window_labels[window_name] for window_name in ndvi_time_order],
    ordered=True,
)
trajectory_plot = trajectory_plot.sort_values(["ecosystem", "window_label"])

fig, axis = plt.subplots(figsize=(6.5, 3.2))
for ecosystem, ecosystem_rows in trajectory_plot.groupby("ecosystem", observed=True):
    axis.plot(
        ecosystem_rows["window_label"].astype(str),
        ecosystem_rows["mean_ndvi_damaged_pixels"],
        marker="o",
        linewidth=1.5,
        label=ecosystem,
    )
axis.axvline(0.5, color="#bdbdbd", linewidth=0.8, linestyle="--")
axis.set_ylabel("Mean NDVI in damaged benefit pixels")
axis.set_title("NDVI trajectory for benefit pixels damaged in months 1-2")
axis.grid(axis="y", color="#e0e0e0", linewidth=0.6)
axis.legend(frameon=False, loc="best")
fig.tight_layout()
trajectory_figure_paths = save_figure(fig, "hurricane_melissa_damaged_nbs_ndvi_trajectory")
plt.show()

trajectory_figure_paths

In [ ]:
category_order = [
    "No improvement or further decline",
    "0-<50% of initial loss recovered",
    "50-<90% of initial loss recovered",
    "90-<100% of initial loss recovered",
    ">=100% of initial loss recovered",
]
category_colors = {
    "No improvement or further decline": "#7f7f7f",
    "0-<50% of initial loss recovered": "#fdae61",
    "50-<90% of initial loss recovered": "#fee08b",
    "90-<100% of initial loss recovered": "#66c2a5",
    ">=100% of initial loss recovered": "#1a9850",
}
category_plot = category_summary.copy()
category_plot["bar_label"] = category_plot["ecosystem"] + "\n" + category_plot["window_label"]
bar_order = [
    "Mangroves\nMonths 3-4",
    "Mangroves\nMonths 5-6",
    "River forest restoration\nMonths 3-4",
    "River forest restoration\nMonths 5-6",
]
pivot = (
    category_plot.pivot_table(
        index="bar_label",
        columns="category",
        values="share_of_damaged_area_pct",
        aggfunc="sum",
    )
    .reindex(bar_order)
    .fillna(0.0)
)
fig, axis = plt.subplots(figsize=(7.0, 3.4))
left_values = np.zeros(len(pivot), dtype="float64")
y_positions = np.arange(len(pivot))
for category_name in category_order:
    values = pivot[category_name].to_numpy() if category_name in pivot.columns else np.zeros(len(pivot))
    axis.barh(
        y_positions,
        values,
        left=left_values,
        color=category_colors[category_name],
        edgecolor="white",
        linewidth=0.4,
        label=category_name,
    )
    left_values += values
axis.set_yticks(y_positions)
axis.set_yticklabels(pivot.index)
axis.set_xlim(0, 100)
axis.set_xlabel("Damaged benefit area (%)")
axis.set_title("Recovery fraction of initial NDVI loss")
axis.grid(axis="x", color="#e0e0e0", linewidth=0.6)
axis.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2)
fig.tight_layout()
category_figure_paths = save_figure(fig, "hurricane_melissa_recovery_fraction_categories")
plt.show()

category_figure_paths

## Save Outputs

In [ ]:
coverage_summary_path = output_dir / "hurricane_melissa_recovery_to_baseline_coverage_summary.csv"
headline_summary_path = output_dir / "hurricane_melissa_recovery_to_baseline_headline_summary.csv"
category_summary_path = output_dir / "hurricane_melissa_recovery_to_baseline_category_summary.csv"
trajectory_summary_path = output_dir / "hurricane_melissa_recovery_to_baseline_trajectory_summary.csv"
metadata_path = output_dir / "hurricane_melissa_recovery_to_baseline_metadata.csv"

coverage_summary.to_csv(coverage_summary_path, index=False)
headline_summary.to_csv(headline_summary_path, index=False)
category_summary.to_csv(category_summary_path, index=False)
trajectory_summary.to_csv(trajectory_summary_path, index=False)

metadata = pd.DataFrame(
    [
        {"name": "pre_event_ndvi", "value": str(ndvi_window_paths["before"])},
        {"name": "early_post_event_damage_state", "value": str(ndvi_window_paths["months_1_2"])},
        {"name": "recovery_months_3_4", "value": str(ndvi_window_paths["months_3_4"])},
        {"name": "recovery_months_5_6", "value": str(ndvi_window_paths["months_5_6"])},
        {"name": "relative_damage_threshold", "value": str(relative_damage_threshold)},
        {"name": "relative_baseline_min", "value": str(relative_baseline_min)},
        {"name": "mangrove_benefit_area_definition", "value": "Mangrove patches with positive avoided EAD in either coastal-flood scenario"},
        {"name": "river_forest_benefit_area_definition", "value": "River-flood forest restoration pixels with positive avoided EAD in either river-flood scenario"},
        {"name": "recovery_fraction_definition", "value": "(NDVI_window - NDVI_months_1_2) / (NDVI_pre_event - NDVI_months_1_2)"},
    ]
)
metadata.to_csv(metadata_path, index=False)

for saved_path in [
    coverage_summary_path,
    headline_summary_path,
    category_summary_path,
    trajectory_summary_path,
    metadata_path,
    *trajectory_figure_paths,
    *category_figure_paths,
]:
    print(saved_path)